# AllDap 채점 모델 파인튜닝: 학습 (M3 + LoRA)

설계: `docs/superpowers/specs/2026-10-05-judge-finetune-design.md` (6절, 7-2절)

**실행 방법**
1. 런타임 > 런타임 유형 변경 > **T4 GPU**. 왼쪽 열쇠 아이콘(보안 비밀)에 이름 `HF_TOKEN` 으로 Hugging Face 토큰을 넣는다.
2. Drive 의 `MyDrive/alldap_judgeft/` 에 `train.jsonl`, `val.jsonl` 을 올린다.
3. **관문 (b)**: 둘째 칸 맨 위를 `MODE = 'gate_b'` 로 고치고 **모두 실행**. 끝나면 `gate_b.json` 이 생긴다. 메모리 부족이나 loss 가 숫자가 아니면 멈추고 알린다(스펙 7-2절).
4. **학습**: 런타임 > 세션 다시 시작 뒤 `MODE = 'train'` 으로 바꾸고 **모두 실행**. 관문 (b) 에서 만든 LoRA 를 이어 쓰지 않도록 반드시 세션을 다시 시작한다. 체크포인트가 하나라도 생긴 뒤에는 `gate_b` 를 다시 돌릴 수 없다.
5. 끊기면 세션을 다시 시작하고 `MODE = 'train'` 그대로 **모두 실행** 한다. 가장 최근의 온전한 체크포인트에서 이어 한다.
6. 끝나면 마지막 칸이 `alldap_judgeft_train.zip`(학습 기록, 어댑터 해시)을 내려받는다. 어댑터 파일은 Drive 의 `adapter/` 에 남는다.

**Drive 휴지통:** 이 노트북은 오래된 체크포인트(하나에 약 0.4GB, 재지 않은 어림)를 지운다. 지운 파일은 Drive 휴지통으로 가서 용량을 계속 차지한다. 학습 기록에 찍히는 `drive_free_gb` 가 3 아래로 내려가면 drive.google.com 의 휴지통에서 **휴지통 비우기** 를 누른다.

설정은 한 번만 학습한다(스펙 6-5절). 이 노트북의 설정 값을 바꾸지 않는다. 시험지는 이 노트북에 없다.

**처음 여는 법:** colab.research.google.com 에서 파일 > 노트북 업로드 로 이 `.ipynb` 파일을 고른다.

**허용 창:** 첫 실행에서 두 창이 뜬다. "이 노트북에 보안 비밀 액세스 권한을 부여하시겠습니까?" 에는 **액세스 권한 부여**, "이 노트북에서 Google Drive 파일에 액세스하도록 허용하시겠습니까?" 에는 **Google Drive에 연결** 을 누르고 계정을 고른 뒤 모두 허용한다. `HF_TOKEN` 이 목록에 있으면 그 줄의 "노트북 액세스" 스위치를 켠다.

**오류가 나면:** 모두 실행이 그 칸에서 멈추고 zip 도 내려받아지지 않는다. 빨간 오류 글의 **마지막 다섯 줄쯤** 을 복사해 그대로 알려 준다. 칸을 고치거나 값을 바꾸지 않는다.

**오래 걸리는 동안:** 브라우저 탭을 닫지 않고 열어 둔다. 컴퓨터의 잠자기를 끈다(맥: 시스템 설정 > 디스플레이 또는 배터리 > 잠자기 방지). 무료 GPU 한도에 걸려 "GPU 백엔드에 연결할 수 없습니다" 가 뜨면 몇 시간 뒤(보통 다음 날) 다시 연다. 이미 한 일은 Drive 에 남아 이어 한다.

In [ ]:
!pip install -q "transformers==4.57.6" "bitsandbytes==0.50.2" "peft==0.17.1" "accelerate==1.10.1" huggingface_hub


In [ ]:
MODE = ''   # 'gate_b' 또는 'train'. 비워 두면 멈춘다(실수로 다른 단계를 돌리지 않게)
assert MODE in ('gate_b', 'train'), "MODE 를 'gate_b' 또는 'train' 으로 고치고 다시 모두 실행하세요."
import gc, glob, hashlib, json, math, os, platform, random, shutil, time
import torch, transformers, bitsandbytes, peft, accelerate
from google.colab import drive, userdata, files
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, GenerationConfig

drive.mount('/content/drive')
DIR = '/content/drive/MyDrive/alldap_judgeft'
HF_TOKEN = userdata.get('HF_TOKEN')
M3_REPO = 'Unbabel/M-Prometheus-14B'
M3_REVISION = 'f790ee744ba08731697e9347c6ab709f9405a37e'   # v2 와 같은 리비전으로 고정한다(스펙 6-1절)
PINS = {'transformers': '4.57.6', 'bitsandbytes': '0.50.2', 'peft': '0.17.1', 'accelerate': '1.10.1'}

def sha_of(path):
    return hashlib.sha256(open(path, 'rb').read()).hexdigest()

def env_record():
    # 판이 하나라도 다르면 같은 설정에서도 출력이 달라질 수 있다(스펙 6-2절). torch 는 코랩에 깔린 것을 적기만 한다.
    env = {'gpu': torch.cuda.get_device_name(0), 'torch': torch.__version__, 'python': platform.python_version(),
           'transformers': transformers.__version__, 'bitsandbytes': bitsandbytes.__version__,
           'peft': peft.__version__, 'accelerate': accelerate.__version__}
    wrong = {k: (v, env[k]) for k, v in PINS.items() if env[k] != v}
    assert not wrong, f'고정한 판과 다르게 깔렸습니다: {wrong}. 런타임을 다시 시작하고 첫 칸부터 다시 실행하세요.'
    return env

def check_env(saved_path):
    # 이어 할 때는 처음 세션과 판이 같아야 한다. 다르면 이어 붙인 결과가 한 환경의 결과가 아니게 된다.
    if os.path.exists(saved_path):
        old = json.load(open(saved_path))
        diff = {k: (old.get(k), ENV.get(k)) for k in set(old) | set(ENV) if old.get(k) != ENV.get(k)}
        assert not diff, f'처음 세션과 환경이 다릅니다: {diff}. 멈추고 이 줄을 그대로 알려 주세요.'
    else:
        json.dump(ENV, open(saved_path, 'w'), indent=2)

QUANT = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
                           bnb_4bit_compute_dtype=torch.float16)  # v2 와 같다. T4 는 bf16 을 지원하지 않는다

def load_base():
    tok = AutoTokenizer.from_pretrained(M3_REPO, revision=M3_REVISION, token=HF_TOKEN)
    # dtype=float16: 4비트로 바뀌지 않는 부분(임베딩, lm_head, 정규화층)의 정밀도. v2 노트북과 같다(스펙 6-1절).
    model = AutoModelForCausalLM.from_pretrained(M3_REPO, revision=M3_REVISION, quantization_config=QUANT,
                                                 dtype=torch.float16, device_map='auto', token=HF_TOKEN)
    return tok, model

TRAIN = f'{DIR}/train.jsonl'
VAL = f'{DIR}/val.jsonl'
EXPECTED = {'train': '000d4c335489921a949b12e5e88cdc97dd97e83d7e16a13f155f7bddcf60e280', 'val': 'e996a7c3e4c2ef6a1dfcc8bfed527ae4411fed5493231fe7060af94b3f5c2e18'}
assert EXPECTED['train'] and EXPECTED['val'], '동결 커밋 전의 노트북입니다. 계획 E11 뒤의 노트북을 쓰세요.'
assert sha_of(TRAIN) == EXPECTED['train'] and sha_of(VAL) == EXPECTED['val'], '올린 학습 데이터의 해시가 다릅니다.'
TRAIN_ROWS = [json.loads(l) for l in open(TRAIN, encoding='utf-8') if l.strip()]
VAL_ROWS = [json.loads(l) for l in open(VAL, encoding='utf-8') if l.strip()]
CKPT = f'{DIR}/checkpoints'
ADAPTER = f'{DIR}/adapter'
LOG = f'{DIR}/training_log.jsonl'
os.makedirs(CKPT, exist_ok=True)
ENV = env_record()

def checkpoints():
    # 완성 표시(done 파일)가 있는 체크포인트만 센다. 저장 도중 끊긴 체크포인트는 done 이 없어 건너뛴다(스펙 6-4절).
    return sorted((int(p.rsplit('-', 1)[1]), p) for p in glob.glob(f'{CKPT}/step-*') if os.path.exists(f'{p}/done'))

def log(rec):
    open(LOG, 'a').write(json.dumps(rec) + '\n')

if MODE == 'gate_b':
    assert not checkpoints(), '학습 체크포인트가 이미 있습니다. 관문 (b) 는 학습 전에만 돌립니다. MODE 를 확인하세요.'
else:
    check_env(f'{DIR}/train_env.json')   # 첫 학습 세션이면 적고, 이어 하는 세션이면 처음과 비교한다
# 세션마다 한 줄. 읽는 쪽은 단계마다 마지막 줄을 쓴다(끊긴 뒤 다시 한 단계가 두 번 적힐 수 있다).
log({'session_start': time.time(), 'mode': MODE, 'env': ENV})
print(len(TRAIN_ROWS), len(VAL_ROWS), ENV)


In [ ]:
from peft import LoraConfig, get_peft_model, set_peft_model_state_dict
from safetensors.torch import load_file

LORA_R, LORA_ALPHA, LORA_DROPOUT = 16, 32, 0.05
LORA_TARGETS = ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj']
LEARNING_RATE, EPOCHS, GRAD_ACCUM, WARMUP_RATIO = 0.0002, 2, 8, 0.03
KEEP_CHECKPOINTS, TRAIN_SEED = 2, 20261006

def build_example(apply_template, encode, messages, target, end_id):
    """(토큰 id 목록, 정답 쪽 길이). 정답 쪽은 모범 답안의 토큰과 끝 토큰(<|im_end|>)이다. 끝 토큰은 맨 뒤에 있다.

    apply_template(messages) 는 대화 틀을 씌운 글(시스템 메시지 포함, 끝에 assistant 차례의 시작)을 돌려준다.
    채점 노트북이 generate 앞에서 만드는 글과 같아야 한다(v2 노트북의 add_generation_prompt=True).
    encode(text) 는 특수 토큰을 더하지 않고 토큰 id 목록을 돌려준다(틀이 이미 특수 토큰을 넣는다).
    입력과 정답을 따로 토큰으로 바꿔 잇는 이유: 채점할 때도 모델은 입력을 따로 토큰으로 바꾼 뒤 그 뒤를 생성한다.
    정답을 입력에 붙여 한꺼번에 바꾸면 경계의 글자가 다르게 묶여, 학습한 토큰 열과 채점 때 생성하는 토큰 열이 달라질 수 있다.
    """
    prompt_ids = list(encode(apply_template(messages)))
    target_ids = list(encode(target)) + [end_id]
    return prompt_ids + target_ids, len(target_ids)

random.seed(TRAIN_SEED); torch.manual_seed(TRAIN_SEED)   # LoRA 초기값을 고정한다
tok, model = load_base()
END_ID = tok.convert_tokens_to_ids('<|im_end|>')
eos = model.generation_config.eos_token_id
assert END_ID in (eos if isinstance(eos, list) else [eos]), '끝 토큰이 생성의 멈춤 토큰이 아니다'
# prepare_model_for_kbit_training 을 쓰지 않는다. 그 함수는 4비트가 아닌 층을 float32 로 바꾸는데, 임베딩과 lm_head 만
# 약 6GB 가 되어 T4 에 들어가지 않을 수 있다. 메모리를 줄이는 두 설정만 직접 켠다.
model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={'use_reentrant': False})
model.enable_input_require_grads()   # gradient checkpointing 에서 입력 쪽 기울기 경로를 연다(LoRA 만 학습해도 필요하다)
model.config.use_cache = False
model = get_peft_model(model, LoraConfig(r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
                                         target_modules=LORA_TARGETS, task_type='CAUSAL_LM'))
for n, p in model.named_parameters():
    if p.requires_grad:
        p.data = p.data.float()   # LoRA 가중치는 float32 로 둔다. float16 으로 고치면 작은 갱신이 0 이 된다(스펙 6-2절)
model.print_trainable_parameters()

def encode_row(row):
    ids, n = build_example(lambda m: tok.apply_chat_template(m, add_generation_prompt=True, tokenize=False),
                           lambda s: tok(s, add_special_tokens=False)['input_ids'], row['messages'], row['target'], END_ID)
    return torch.tensor(ids, device='cuda'), n

def row_loss(row):
    ids, n = encode_row(row)
    with torch.autocast('cuda', dtype=torch.float16):
        # logits_to_keep=n+1: 마지막 n+1 위치의 logits 만 계산한다. 모범 답안의 토큰 n 개를 맞히는 데 필요한 것이
        # 그 앞 위치들이기 때문이다. 입력 위치의 logits(약 15만 어휘 곱하기 입력 길이)를 만들지 않아 메모리가 준다.
        out = model(input_ids=ids[None], logits_to_keep=n + 1)
    logits = out.logits[0, :-1].float()   # n 개 위치. float32 로 바꿔 loss 를 계산한다
    return torch.nn.functional.cross_entropy(logits, ids[-n:])

TOTAL_STEPS = EPOCHS * math.ceil(len(TRAIN_ROWS) / GRAD_ACCUM)
WARMUP = max(1, round(WARMUP_RATIO * TOTAL_STEPS))
opt = bitsandbytes.optim.PagedAdamW8bit([p for p in model.parameters() if p.requires_grad], lr=LEARNING_RATE)
sched = transformers.get_cosine_schedule_with_warmup(opt, WARMUP, TOTAL_STEPS)
scaler = torch.amp.GradScaler('cuda')   # float16 혼합 정밀도의 loss scaling(스펙 6-2절)
print('학습 단계', TOTAL_STEPS, 'warmup', WARMUP)

def batches():
    """학습 단계마다 문항 묶음. 바퀴마다 시드로 순서를 섞는다. 같은 시드면 끊겼다 이어 해도 같은 순서다."""
    for e in range(EPOCHS):
        order = random.Random(f'{TRAIN_SEED}:{e}').sample(range(len(TRAIN_ROWS)), len(TRAIN_ROWS))
        for i in range(0, len(order), GRAD_ACCUM):
            yield [TRAIN_ROWS[j] for j in order[i:i + GRAD_ACCUM]]

def train_step(batch):
    losses = []
    for row in batch:
        loss = row_loss(row) / len(batch)
        if not torch.isfinite(loss):
            raise RuntimeError(f'loss 가 숫자가 아니다({loss.item()}). 설정을 바꾸지 않고 멈춘다(스펙 6-5절).')
        scaler.scale(loss).backward()
        losses.append(loss.item() * len(batch))
    scale = scaler.get_scale()
    scaler.step(opt); scaler.update(); sched.step(); opt.zero_grad(set_to_none=True)
    # 기울기에 inf, NaN 이 있으면 scaler 는 그 단계의 갱신을 건너뛰고 scale 을 줄인다. 줄었으면 건너뛴 것이다.
    # 약 135단계 중 warmup 이 4단계라 앞쪽 건너뜀이 많으면 학습이 덜 된 채 끝날 수 있어 기록한다.
    return sum(losses) / len(losses), scale, scaler.get_scale() < scale


In [ ]:
if MODE == 'gate_b':
    # from_pretrained 는 모델을 eval 모드로 돌려주고 get_peft_model 은 바깥 껍데기만 training=True 로 둔다.
    # 층이 eval 이면 gradient checkpointing 이 꺼져(층의 조건이 gradient_checkpointing and training) 메모리를 더 쓴다.
    # 그러면 관문이 메모리 부족으로 거짓 실패하거나 step_seconds, est_hours, save_every 가 학습과 다르게 나온다.
    model.train()
    layers = model.base_model.model.model.layers
    assert all(l.gradient_checkpointing and l.training for l in layers), 'gradient checkpointing 이 켜지지 않았습니다'
    lengths = [(len(encode_row(r)[0]), r['id']) for r in TRAIN_ROWS + VAL_ROWS]
    longest_tokens, longest_id = max(lengths)
    row = next(r for r in TRAIN_ROWS + VAL_ROWS if r['id'] == longest_id)
    torch.cuda.reset_peak_memory_stats()
    times, losses = [], []
    for _ in range(2):   # 학습 단계 둘. 첫 단계는 준비 시간이 섞이므로 둘째 단계의 시간을 쓴다
        t = time.time(); losses.append(train_step([row] * GRAD_ACCUM)[0]); torch.cuda.synchronize(); times.append(time.time() - t)
        assert all(l.gradient_checkpointing and l.training for l in layers), '첫 단계 뒤 gradient checkpointing 이 꺼졌습니다'
    step_seconds = times[-1]
    res = {'longest_id': longest_id, 'longest_tokens': longest_tokens, 'losses': losses, 'env': ENV,
           'finite': all(math.isfinite(x) for x in losses),
           'peak_gb': round(torch.cuda.max_memory_allocated() / 1024**3, 2), 'step_seconds': round(step_seconds, 1),
           'total_steps': TOTAL_STEPS, 'est_hours': round(step_seconds * TOTAL_STEPS / 3600, 1),
           'save_every': max(1, int(1800 // step_seconds))}
    json.dump(res, open(f'{DIR}/gate_b.json', 'w'), indent=2)
    print(res)
    assert res['finite'], 'loss 가 숫자가 아니다. 멈추고 사용자에게 알린다(스펙 7-2절)'


In [ ]:
def val_loss():
    model.eval()
    with torch.no_grad():
        v = sum(row_loss(r).item() for r in VAL_ROWS) / len(VAL_ROWS)
    model.train()
    return v

FILES = ('adapter_model.safetensors', 'state.pt')

def drive_free_gb():
    # Drive 를 붙인 폴더의 남은 용량. Drive 가 알려 주는 값이라 휴지통이 차지한 몫이 빠져 있을 수 있다(재지 않음).
    return round(shutil.disk_usage(DIR).free / 1024**3, 1)

def save(step):
    p = f'{CKPT}/step-{step}'
    shutil.rmtree(p, ignore_errors=True)   # 같은 번호의 깨진 체크포인트가 남아 있으면 그 done 이 새 것으로 보이지 않게 지운다
    os.makedirs(p)
    model.save_pretrained(p)
    torch.save({'opt': opt.state_dict(), 'sched': sched.state_dict(), 'scaler': scaler.state_dict(), 'step': step,
                'py': random.getstate(), 'torch': torch.get_rng_state(), 'cuda': torch.cuda.get_rng_state()}, f'{p}/state.pt')
    # done 은 마지막에 쓴다. 이 파일이 있어야 완성된 체크포인트다. 두 파일의 해시를 적어 이어 할 때 깨졌는지 본다.
    json.dump({f: sha_of(f'{p}/{f}') for f in FILES}, open(f'{p}/done', 'w'))
    for _, old in checkpoints()[:-KEEP_CHECKPOINTS]:
        shutil.rmtree(old)   # 가장 최근 둘만 남긴다(Drive 15GB, 스펙 6-4절). 지운 파일은 Drive 휴지통으로 간다

def load_checkpoint(p):
    """해시가 맞고 읽히면 단계 번호를, 아니면 None 을 돌려준다. 읽다가 실패하면 모델이 반쯤 바뀌었을 수 있어 멈춘다."""
    want = json.load(open(f'{p}/done'))
    got = {f: sha_of(f'{p}/{f}') for f in FILES}
    if got != want:
        return None
    set_peft_model_state_dict(model, load_file(f'{p}/adapter_model.safetensors'))
    s = torch.load(f'{p}/state.pt', weights_only=False)
    opt.load_state_dict(s['opt']); sched.load_state_dict(s['sched']); scaler.load_state_dict(s['scaler'])
    random.setstate(s['py']); torch.set_rng_state(s['torch']); torch.cuda.set_rng_state(s['cuda'])
    return s['step']

if MODE == 'train':
    SAVE_EVERY = json.load(open(f'{DIR}/gate_b.json'))['save_every']
    start = 0
    # 가장 최근 것부터 본다. 해시가 맞지 않거나 파일을 읽지 못하면 기록하고 지운 뒤 하나 앞의 것으로 간다.
    # 최근 둘을 남기는 이유(스펙 6-4절)를 코드가 실제로 쓰는 자리다.
    for n, p in reversed(checkpoints()):
        try:
            got = load_checkpoint(p)
            why = None if got is not None else '해시가 done 에 적힌 것과 다르다'
        except Exception as e:   # 파일이 잘려 safetensors 나 torch.load 가 실패한 경우
            raise RuntimeError(f'체크포인트 {p} 를 읽다가 멈췄습니다({e}). 모델 상태가 섞였을 수 있어 세션을 다시 시작하고 '
                               f'이 체크포인트 폴더를 지운 뒤 다시 모두 실행하세요.') from e
        if why is None:
            start = got
            log({'resumed_from': start, 'at': time.time(), 'env': ENV})
            print('이어 한다:', p)
            break
        log({'bad_checkpoint': n, 'reason': why, 'at': time.time()})
        print('깨진 체크포인트를 건너뛴다:', p, why)
        shutil.rmtree(p)
    model.train()
    for step, batch in enumerate(batches(), 1):
        if step <= start:
            continue   # 이미 한 단계는 건너뛴다. batches() 는 시드로 순서가 정해져 있어 같은 자리에서 이어진다
        loss, scale, skipped = train_step(batch)
        rec = {'step': step, 'train_loss': loss, 'lr': sched.get_last_lr()[0], 'scale': scale, 'skipped': skipped,
               'at': time.time()}
        if step % SAVE_EVERY == 0 or step == TOTAL_STEPS:
            rec['val_loss'] = val_loss()
            save(step)
            rec['drive_free_gb'] = drive_free_gb()
        log(rec)
        print(rec)
    model.save_pretrained(ADAPTER)   # 채점에 쓰는 어댑터는 마지막 단계의 것이다(스펙 6-3절). 검증 loss 로 고르지 않는다
    sha = sha_of(f'{ADAPTER}/adapter_model.safetensors')
    json.dump({'adapter_sha256': sha, 'steps': TOTAL_STEPS, 'env': ENV}, open(f'{DIR}/train_done.json', 'w'), indent=2)
    print('어댑터 SHA-256:', sha)


In [ ]:
os.makedirs('/content/train_out', exist_ok=True)
for name in ('gate_b.json', 'training_log.jsonl', 'train_done.json', 'train_env.json'):
    if os.path.exists(f'{DIR}/{name}'):
        shutil.copy(f'{DIR}/{name}', '/content/train_out/')
shutil.make_archive('/content/alldap_judgeft_train', 'zip', '/content/train_out')
files.download('/content/alldap_judgeft_train.zip')
